::: {.sb-lesson-links}
[Exercises →](../exercises/electricity.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/08-time-series.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-time-series.ipynb){download="lesson-time-series.ipynb"}  
Data: [apple_stock.csv](../downloads/data/apple_stock.csv){download="apple_stock.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

## Introduction
What is the essential difference between time series methods and classic machine learning methods?
Time series analysis takes into account that data points collected over time may have an internal structure (such as autocorrelation, a trend or seasonal variation).
This lecture gives a brief overview of some of the most widely used techniques in the rich and rapidly growing field of time series modeling and analysis. The next lecture gives applied examples with the Prophet library.

*Definition of Time Series:* An ordered sequence of values of a variable at equally spaced time intervals.

*Applications:* Time series models have two uses:

* Understand the underlying forces and structure that produced the observed data
* Fit a model, then use it for forecasting, monitoring or even feedback and feedforward control.

Time series analysis is used in many fields, such as:

* Economic Forecasting
* Sales Forecasting
* Budgetary Analysis
* Stock Market Analysis
* Yield Projections
* Process and Quality Control
* Inventory Studies
* Workload Projections
* Utility Studies
* Census Analysis
* and many, many more...

*Techniques:* Fitting a time series model can be an ambitious task. There are many methods to model and forecast time series, including:

* Box-Jenkins ARIMA models
* Box-Jenkins Multivariate Models
* Holt-Winters Exponential Smoothing (single, double, triple)

## Smoothing techniques

Data collected over time always contains some random variation. Some methods reduce or cancel the effect of this random variation. A common one is "smoothing": when properly applied, it reveals more clearly the underlying trend, seasonal and cyclic components.

### Averaging

To predict how a metric evolves over time, we can use averaging techniques: they predict future values, or estimate the general trend of the series, with some kind of average of the measured values.
For example, let's consider the following dataset (the Apple stock price in 2019):


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore") # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"


def plot_smoothing(x, original, smoothed, name, title, x_label="date"):
    """The original series (grey) and a smoothed or predicted series (blue) on the same figure."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=original, mode="lines", name="opening price", line=dict(color="#999999")))
    fig.add_trace(go.Scatter(x=smoothed.index, y=smoothed, mode="lines", name=name, line=dict(color="#3B47C4", width=2.5)))
    fig.update_layout(title=title, xaxis_title=x_label, yaxis_title="price ($)", legend=dict(orientation="h", y=1.1))
    return fig


def plot_correlogram(values, n, title):
    """Correlations from lag 1 (lag 0 is always 1), with the 95% significance limits (dashed)."""
    lags = np.arange(1, len(values))
    limit = 1.96 / np.sqrt(n)
    fig = go.Figure(go.Bar(x=lags, y=values[1:], marker_color="#3B47C4"))
    for level in (limit, -limit):
        fig.add_hline(y=level, line_dash="dash", line_color="#E4572E")
    fig.update_layout(title=title, xaxis_title="lag (trading days)", yaxis_title="correlation", showlegend=False)
    return fig

In [ ]:
# Get the data of the stock AAPL
data = pd.read_csv("../data/apple_stock.csv", index_col="Date", parse_dates=True).loc["2019-01-01":"2020-01-01"]
data

,Open,High,Low,Close,Volume
Date,,,,,
2019-01-02,36.6907,37.6288,36.5344,37.4085,148158800
2019-01-03,34.1063,34.5185,33.6373,33.6823,365248800
2019-01-04,34.2366,35.1889,34.0637,35.1202,234428400
2019-01-07,35.2244,35.2552,34.5612,35.0420,219111200
2019-01-08,35.4282,35.9635,35.1818,35.7100,164101200
...,...,...,...,...,...
2019-12-24,68.4513,68.4994,68.0258,68.3504,48478800
2019-12-26,68.4826,69.7233,68.4538,69.7065,93121200
2019-12-27,69.9974,70.6827,69.2761,69.6800,146266000


Let's create a new column that contains the average of the variable ```Open```, plot both columns on the same graph, and compute the MSE between them.

In [ ]:
data["open_mean"] = data["Open"].mean()

fig = plot_smoothing(data.index, data["Open"], data["open_mean"], "global average", "Global average smoothing")
fig.show()

MSE = np.mean((data["Open"] - data["open_mean"]) ** 2)
print(f"The MSE of the global average smoothing is: {MSE:.2f}")

The MSE of the global average smoothing is: 70.64


Of course, the global average is a rather naive smoothing technique to estimate the characteristics of a time series or to forecast future values. Here the data shows a positive growth trend: the further we look from the middle of the period, the larger the errors made by this method.

### Single moving average

Another way to summarize past data is to compute the mean over successive, smaller windows of past values:

$$
M_t = \frac{X_t+X_{t-1}+ ... +X_{t-T+1}}{T}
$$
where $t$ is the observed date, $X_t$ is the value of the time series at date $t$, and $T$ is the number of dates covered by the moving average (also called the order of the moving average).

Let's now compute the moving average of order 7 on the ```Open``` column of our dataset (7 trading days, about a week and a half).


In [ ]:
y = data["Open"]
data["Open_MA_7"] = y.rolling(window=7).mean()

fig = plot_smoothing(data.index, data["Open"], data["Open_MA_7"], "moving average (7 days)", "Moving average of order 7")
fig.show()

As you can see, a moving average follows the general trend of the data much better, depending on the chosen order (try different orders and look at the effect on the plot). However, moving averages have a major drawback:
since the moving average is computed over the $T$ previous dates (current date included), it always lags behind the actual time series, and the higher the order, the longer the lag.

**See it with data 🔬 Choosing the window of a moving average**

The order $T$ is a trade-off. Let's put three windows on the same plot (3, 7 and 30 trading days) and measure the MSE between each moving average and the opening price.

In [ ]:
fig = go.Figure(go.Scatter(x=data.index, y=y, mode="lines", name="opening price", line=dict(color="#999999")))
for window, color in [(3, "#2E7D4F"), (7, "#3B47C4"), (30, "#E4572E")]:
    smoothed = y.rolling(window=window).mean()
    mse = ((y - smoothed) ** 2).mean()                        # the first window-1 days (NaN) are skipped
    print(f"window = {window:2d} trading days: MSE = {mse:.2f}")
    fig.add_trace(go.Scatter(x=data.index, y=smoothed, mode="lines", name=f"window {window} (MSE {mse:.2f})",
                             line=dict(color=color, width=2)))
fig.update_layout(title="Moving averages of order 3, 7 and 30", xaxis_title="date", yaxis_title="price ($)",
                  legend=dict(orientation="h", y=1.1))
fig.show()

window =  3 trading days: MSE = 0.27
window =  7 trading days: MSE = 0.95
window = 30 trading days: MSE = 7.56


A window of 3 days stays very close to the data (MSE 0.27) but keeps most of the day-to-day noise. A window of 30 days gives a very smooth curve, but it lags well behind the rising price (MSE 7.56, almost 30 times more) and it only starts after 30 days. The window of 7 (MSE 0.95) is in between. A small MSE is not the goal in itself, otherwise $T = 1$ (the data itself) would win: we choose $T$ for how much noise we want to remove.

### Exponential Smoothing

This is a very popular way to smooth a time series. In a single moving average, the past observations all have the same weight; exponential smoothing gives them weights that decrease exponentially as the observations get older.

In other words, recent observations weigh more in the forecast than older ones.

#### Single exponential smoothing

This method starts by setting $S_2$ to $X_1$, where $S_i$ is the smoothed observation at date $i$.



::: {.callout-tip title="Intuition"}
Each day, the smoothed value moves a fraction $\alpha$ of the way from its previous value towards the new observation. With $\alpha$ close to 1, you mostly trust today's value; with $\alpha$ close to 0, you mostly trust your past estimate, so the curve is smooth but slow to react. Unrolling the formula shows that an observation $k$ dates old gets the weight $\alpha(1-\alpha)^k$: this is where the name "exponential" comes from.
:::



For any date $t$, the smoothed value $S_t$ is given by:

$$
S_t = \alpha X_t + (1-\alpha)S_{t-1}
$$

where $\alpha \in [0,1]$ and $t \geq 3$.

Let's compute an exponential moving average of our column ```Open``` with $\alpha=0.1$:

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

# statsmodels needs a regular index: we number the trading days 0, 1, 2...
SES = SimpleExpSmoothing(data["Open"].reset_index(drop=True), initialization_method="estimated")
SES_fit = SES.fit(smoothing_level=0.1)
SES_predict = SES_fit.predict(start=0, end=len(data) + 30)

fig = plot_smoothing(np.arange(len(data)), data["Open"], SES_predict, "simple exponential smoothing (α = 0.1)",
                     "Simple exponential smoothing, and 30 days of forecast", x_label="trading day")
fig.show()

Exponential smoothing smooths the time series really well, but it has two major flaws:

1. the smoothed series lags behind the original series
2. when we predict future values, as we did here, the forecast is a flat line

**See it with data 🔬 The effect of α**

Let's smooth the same series with $\alpha$ = 0.1, 0.5 and 0.9. pandas' `ewm(alpha=..., adjust=False)` applies the recursive formula $S_t = \alpha X_t + (1-\alpha)S_{t-1}$ directly.

In [ ]:
fig = go.Figure(go.Scatter(x=data.index, y=y, mode="lines", name="opening price", line=dict(color="#999999")))
for alpha, color in [(0.1, "#E4572E"), (0.5, "#3B47C4"), (0.9, "#2E7D4F")]:
    smoothed = y.ewm(alpha=alpha, adjust=False).mean()
    mse = ((y - smoothed) ** 2).mean()
    print(f"alpha = {alpha}: MSE = {mse:.3f}")
    fig.add_trace(go.Scatter(x=data.index, y=smoothed, mode="lines", name=f"α = {alpha} (MSE {mse:.3f})",
                             line=dict(color=color, width=2)))
fig.update_layout(title="Single exponential smoothing with three values of α", xaxis_title="date",
                  yaxis_title="price ($)", legend=dict(orientation="h", y=1.1))
fig.show()

alpha = 0.1: MSE = 3.070
alpha = 0.5: MSE = 0.171
alpha = 0.9: MSE = 0.005


With $\alpha = 0.9$ the smoothed curve almost copies the data (MSE 0.005): nearly no smoothing. With $\alpha = 0.1$ it is very smooth but clearly lags below the rising price (MSE 3.070). $\alpha = 0.5$ (MSE 0.171) is a compromise. $\alpha$ plays the same role as the window of the moving average, in reverse: a small $\alpha$ is like a long window.

#### Double exponential smoothing

As we just saw, single smoothing does not follow the data well when there is a trend. We can improve this by adding a second equation with a second constant, $\gamma$, chosen together with $\alpha$.

Here are the two equations of double exponential smoothing:

$$
S_t = \alpha X_t + (1-\alpha)(S_{t-1} + b_{t-1}) \; \alpha \in [0,1]
$$

$$
b_t = \gamma(S_t - S_{t-1}) + (1-\gamma)b_{t-1} \; \gamma \in [0,1]
$$
Note that the current value of the series, $X_t$, is used to compute its smoothed value $S_t$, and $b_t$ estimates the trend (the slope).
Let's compute the double exponential smoothing of our ```Open``` series.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
DES = ExponentialSmoothing(data["Open"].reset_index(drop=True), trend="add")
DES_fit = DES.fit()
DES_predict = DES_fit.predict(start=0, end=len(data) + 30)

fig = plot_smoothing(np.arange(len(data)), data["Open"], DES_predict, "double exponential smoothing",
                     "Double exponential smoothing (level + trend), and 30 days of forecast", x_label="trading day")
fig.show()

What strikes us here is that the lag is much smaller (though still present), and the future predictions now follow a positive trend, like our time series! However, our predictions do not show the ups and downs that we would expect from the data.

#### Triple exponential smoothing

What happens if the data show trend and seasonality?

In this case, double smoothing will not work. We add a third equation to handle seasonality (sometimes called periodicity). The resulting set of equations is called the "Holt-Winters" (HW) method, after the names of its inventors.
The equations of the additive version (the season is *added* to the level; in the multiplicative version, it multiplies it) are:

$$
S_t = \alpha (X_t - I_{t-T}) + (1-\alpha)(S_{t-1} + b_{t-1}) \textrm{  OVERALL SMOOTHING}
$$

$$
b_t = \gamma(S_t - S_{t-1}) + (1-\gamma)b_{t-1} \textrm{  TREND SMOOTHING}
$$

$$
I_t = \beta (X_t - S_t) + (1-\beta)I_{t-T} \textrm{  SEASONAL SMOOTHING}
$$

$$
F_{t+m} = S_t + m b_t + I_{t-T+m} \textrm{  FORECAST}
$$

where:

* $X$ is the observation
* $S$ is the smoothed observation
* $b$ is the trend factor
* $I$ is the seasonal index
* $F$ is the forecast at $m$ dates ahead
* $t$ is the date
* $T$ is the period of the potential seasonality

Let's now predict our time series with triple exponential smoothing, also known as the Holt-Winters model (in the code we use its additive version, `seasonal='add'`, with a period of 20 trading days, about a month):



In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
TES = ExponentialSmoothing(y.reset_index(drop=True), trend="add", seasonal="add", seasonal_periods=20)
TES_fit = TES.fit()
TES_predict = TES_fit.predict(start=0, end=len(data) + 60)

fig = plot_smoothing(np.arange(len(data)), data["Open"], TES_predict, "Holt-Winters (triple exponential smoothing)",
                     "Holt-Winters (level + trend + season of 20 days), and 60 days of forecast", x_label="trading day")
fig.show()

A slight lag is still present, but this time our forecasts have both a trend and a seasonality, so the predictions for the first few dates look believable. After a while, however, the predictions repeat the same pattern again and again: the Holt-Winters model is not a good candidate for long-term predictions (here, long term means further into the future than one seasonal period).

## Univariate time series modeling

The term "univariate time series" refers to a single variable measured over time.

### Stationarity
A common assumption in many time series techniques is that the data are stationary.

A stationary process has the property that the mean, variance and autocorrelation structure do not change over time. Stationarity can be defined in precise mathematical terms, but here we simply mean a flat-looking series: no trend, a constant variance over time, a constant autocorrelation structure over time and no periodic fluctuations (seasonality).

Here's an example of a stationary time series:

In [ ]:
sample = np.random.default_rng(0).standard_normal(200)
fig = px.line(x=range(200), y=sample, labels={"x": "time", "y": "value"}, title="White noise: 200 draws of N(0, 1)")
fig.show()

### Transformations to Achieve Stationarity
If the time series is not stationary, we can often make it stationary with one of the following techniques:

* We can difference the data. That is, given the series $X_t$, we create the new series
$X'_t=X_t−X_{t−1}$.
The differenced data contains one point less than the original data. You can difference the data more than once, but one difference is usually enough.

* If the data contains a trend, we can fit a curve to the data and then model the residuals of that fit. Since the fit only aims at removing the long-term trend, a simple fit, such as a straight line, is usually enough.

* For non-constant variance, taking the logarithm or square root of the series may stabilize the variance. For negative data, you can add a suitable constant to make all the data positive before the transformation. You then subtract this constant from the model outputs to obtain the predicted (i.e., fitted) values and the forecasts of future points.

These techniques aim at producing a series with a constant location (mean) and scale (variance). Seasonality also breaks stationarity, but it is usually included explicitly in the time series model.

### White noise

A white noise is a discrete-time stochastic process $A_t$ whose mean does not depend on the time $t$ and is equal to zero, i.e. $E[A_t]=0$, and whose autocorrelation function $E[A_t A_{t+k}]$ only depends on $k$ (not on $t$) and is nonzero only for $k=0$.
The stationary series plotted above is an example of white noise: it is made of random samples drawn from a $N(0,1)$ normal distribution.

### Autoregressive (AR) Models

A common approach for modeling univariate time series is the autoregressive (AR) model:

$$
X_t = \delta + \phi_1 X_{t-1} + \phi_2 X_{t-2} + ... + \phi_{p}X_{t-p} + A_t
$$

where $X_t$ is the time series, $A_t$ is white noise, and
$\delta=(1−\sum_{i=1}^{p}\phi_i)\mu$,
with $\mu$ denoting the process mean.

An autoregressive model is simply a linear regression of the current value of the series against one or more prior values of the series. The value of $p$ is called the order of the AR model.

AR models can be fitted with various methods, including standard linear least squares. They are also easy to interpret.

### Moving Average Models

Another common approach for modeling univariate time series is the moving average (MA) model:

$$
X_t = \mu + A_t - \theta_1 A_{t-1} -\theta_2 A_{t-2} - ... - \theta_q A_{t-q}
$$

where $X_t$ is the time series, $\mu$ is the mean of the series, $A_{t−i}$ are white noise terms, and $\theta_1,…,\theta_q$ are the parameters of the model. The value of $q$ is called the order of the MA model.

In other words, a moving average model is conceptually a linear regression of the current value of the series against the white noise, or random shocks, of one or more prior values of the series. The random shocks at each point are assumed to come from the same distribution, typically a normal distribution with mean zero and constant variance. What makes this model special is that these random shocks propagate to future values of the time series.

(Do not confuse the MA model with the moving average smoothing seen above: they share a name, not a formula.)

### Box-Jenkins ARIMA
The Box-Jenkins ARIMA model combines the AR and MA models:

$$
D_t = \delta + \phi_1 D_{t-1} + \phi_2 D_{t-2} + ... + \phi_{p}D_{t-p} + A_t - \theta_1 A_{t-1} -\theta_2 A_{t-2} - ... - \theta_q A_{t-q}
$$

where the terms have the same meaning as in the AR and MA models, and $D_t$ is the original time series $X_t$, possibly differenced.

The I in ARIMA stands for "integrated": the model takes into account how many times the series was differenced, so it can go from $D_t$ back to $X_t$. Remember that AR and MA models only work well on stationary time series (and they do not handle seasonality), so $X_t$ may have to be differenced to become stationary.

### Box-Jenkins Model Identification

#### Stationarity and Seasonality
The first step in building a Box-Jenkins model is to check whether the series is stationary, and whether there is a significant seasonality to model.

#### Detecting stationarity
Stationarity can be assessed from a run sequence plot (the series plotted over time), which should show a constant location and scale. It can also be detected from an autocorrelation plot: an autocorrelation that decays very slowly is often a sign of non-stationarity.

#### Detecting seasonality
Seasonality (or periodicity) can usually be assessed from an autocorrelation plot, a seasonal subseries plot, or a spectral plot.

#### Differencing to achieve stationarity
Box and Jenkins recommend differencing to achieve stationarity. However, fitting a curve and subtracting the fitted values from the original data also works with Box-Jenkins models.

#### Seasonal differencing
At the model identification stage, our goal is to detect seasonality, if it exists, and to identify the order for the seasonal autoregressive and seasonal moving average terms. For many series, the period is known and a single seasonality term is sufficient. For example, for monthly data we would typically include either a seasonal AR 12 term or a seasonal MA 12 term. For Box-Jenkins models, we do not explicitly remove seasonality before fitting the model. Instead, we give the order of the seasonal terms to the ARIMA estimation software, as part of the model specification. However, it may help to apply a seasonal difference to the data and plot the autocorrelation and partial autocorrelation again: this can help identify the non-seasonal part of the model. In some cases, seasonal differencing removes most or all of the seasonality.

#### Identify p and q
Once stationarity and seasonality have been handled, the next step is to identify the orders (i.e., $p$ and $q$) of the autoregressive and moving average terms. The table below gives the shape of the autocorrelation plot and the model it suggests.

| SHAPE | INDICATED MODEL |
| --- | --- |
|Exponential, decaying to zero | Autoregressive model. Use the partial autocorrelation plot to identify the order of the autoregressive model. |
| Alternating positive and negative, decaying to zero | Autoregressive model. Use the partial autocorrelation plot to help identify the order. |
| One or more spikes, rest are essentially zero | Moving average model, order identified by where the plot becomes zero. |
| Decay, starting after a few lags | Mixed autoregressive and moving average model.|
|All zero or close to zero | Data is essentially random.|
| High values at fixed intervals  | Include seasonal autoregressive term.|
| No decay to zero | Series is not stationary.|

#### Autocorrelation

Autocorrelation measures the correlation of the time series with itself, shifted by a certain lag. For example, the autocorrelation of order $p$ of a series $X$ is:

$$
\rho_p = \frac{E[(X_t - \bar{X})(X_{t-p}-\bar{X})]}{\sigma^2}
$$
where $\sigma^2$ is the variance of the series and $\bar{X}$ its mean (often 0 for the stationary series we work with).

Let's look at the autocorrelation plot of our ```Open``` series.


In [ ]:
import statsmodels.api as sm
Open = data["Open"]

autocor = sm.tsa.stattools.acf(Open)
fig = plot_correlogram(autocor, len(Open), "Autocorrelation of the opening price")
fig.show()

The autocorrelation decays very slowly: the series is not stationary, because we have not removed the trend from the data. We can do this by differencing the series once.

In [ ]:
Open_diff = Open.diff()[1:]
autocor = sm.tsa.stattools.acf(Open_diff)

fig = px.line(x=Open_diff.index, y=Open_diff, labels={"x": "date", "y": "change ($)"},
              title="Daily change of the opening price (differenced series)")
fig.show()

fig = plot_correlogram(autocor, len(Open_diff), "Autocorrelation of the daily changes")
fig.show()

Here, apart from lag 0 (always equal to 1), no autocorrelation is significant at the 95% level (the significance limits are the two horizontal lines). So the daily changes might be purely random.

**See it with data 🔬 Is the differenced series really stationary?**

Two quick checks. First, compare the mean and standard deviation of the first and second halves of the year: for a stationary series they should be similar. Second, the *Augmented Dickey-Fuller* (ADF) test from statsmodels: its starting hypothesis is "the series is **not** stationary", so a small p-value (below 0.05) is evidence that the series *is* stationary.

In [ ]:
from statsmodels.tsa.stattools import adfuller

for name, series in [("opening price", Open), ("daily changes", Open_diff)]:
    half = len(series) // 2
    first, second = series.iloc[:half], series.iloc[half:]
    print(f"{name}:\n  mean: {first.mean():6.2f} (1st half) vs {second.mean():6.2f} (2nd half)"
          f"   std: {first.std():.2f} vs {second.std():.2f}   ADF p-value: {adfuller(series)[1]:.3g}")

opening price:
  mean:  43.42 (1st half) vs  55.95 (2nd half)   std: 4.16 vs 6.78   ADF p-value: 0.988
daily changes:
  mean:   0.09 (1st half) vs   0.17 (2nd half)   std: 0.67 vs 0.71   ADF p-value: 3.79e-29


The raw price is not stationary: its mean jumps from 43.42 to 55.95 dollars between the first and the second half of the year, and the ADF p-value is 0.988, so we can't reject "not stationary". The daily changes have almost the same mean (0.09 vs 0.17 dollars) and standard deviation (0.67 vs 0.71) in both halves, and an ADF p-value of about $4\times10^{-29}$: one differencing was enough to make the series stationary.

#### Partial autocorrelations
The problem with autocorrelation is that the autocorrelation of order 3 also contains the indirect correlations of order 2 and 1. Indeed, if $X_t$ and $X_{t-1}$ are autocorrelated, then $X_{t-3}$ is correlated with $X_{t-2}$, which is correlated with $X_{t-1}$, which is correlated with $X_t$. Partial autocorrelation measures the direct correlation between $X_t$ and $X_{t-p}$, once the effect of the lower orders is removed. The partial autocorrelation of order $q$ is written $\alpha_q$.

In [ ]:
pautocor = sm.tsa.stattools.pacf(Open)
fig = plot_correlogram(pautocor, len(Open), "Partial autocorrelation of the opening price")
fig.show()

This is the partial autocorrelation plot of the original series. It is not stationary, so let's plot the partial autocorrelation of the differenced series:

In [ ]:
pautocor = sm.tsa.stattools.pacf(Open_diff)
fig = plot_correlogram(pautocor, len(Open_diff), "Partial autocorrelation of the daily changes")
fig.show()

The conclusion: the daily changes of the stock price look random over time, which is not really surprising for stocks.

## Conclusion

You now know the most standard analysis tools for univariate time series. For a practical approach to time series modeling, move on to the next lecture, "Time series with Prophet": it shows you how to build a time series model very easily with Prophet.

::: {.sb-next}
**[Next: Forecasting with Prophet →](../lessons/prophet.ipynb)**
:::